# Halo exchange AD across processes

`nb01_halo_exchange_adjoint.ipynb` derived the adjoint of a halo exchange and checked
it three ways. All of it happened in one process, on one array, where owner and halo
are neighbouring slices of the same buffer and the reverse exchange is a local `+=`.

Distributing the model changes exactly one thing, but it is the thing that matters:
**the accumulation now has to cross a process boundary.** The reverse exchange becomes
a real communication, running in the opposite direction from the forward one and
reducing instead of assigning.

There are two ways to get a differentiable distributed halo exchange, and they are
genuinely different projects:

- **Route A — `shard_map` + `lax.ppermute`.** Stay inside JAX. The collective is a
  linear primitive JAX already knows how to transpose, so the adjoint costs nothing.
- **Route B — MPI + `jax.custom_vjp`.** Go outside JAX. The exchange is opaque, so
  you supply the backward pass yourself — and nb01 is its specification.

Both are checked here against the same single-process reference.

## 0. Setup

`XLA_FLAGS` has to be set before JAX initialises its backend, so this must be the
first cell you run. Four CPU devices stand in for four ranks.

In [1]:
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=4"

import subprocess

import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_enable_x64", True)

from jax.sharding import Mesh, PartitionSpec as P

try:
    from jax.experimental.shard_map import shard_map
except ImportError:
    from jax.sharding import shard_map

from gt4py import next as gtx
from gt4py.next.experimental import concat_where
from operators import I, J, IJField
from adjoint_operators import halo_exchange, halo_domain, interior_domain

D = 4                      # ranks along I
MLOC, N = 3, 5             # per-rank interior extent in I; full extent in J
M = D * MLOC               # global interior extent in I

print(f"jax {jax.__version__}")
print("devices:", jax.devices())


def fld(domain, array):
    return gtx.as_field(domain, jnp.asarray(array, dtype=jnp.float64), allocator=jnp)

jax 0.6.2
devices: [CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3)]


## 1. The reference

A halo exchange on its own is easy to get accidentally right, because if nothing reads
the halo the adjoint of the halo is zero either way. So the reference pipeline puts a
five-point stencil *after* the exchange — now every halo value feeds an interior
output, and any error in the exchange adjoint shows up in the gradient.

    interior (M, N)  ->  halo exchange  ->  five-point stencil  ->  interior (M, N)

In [2]:
@gtx.field_operator
def periodic_j(f: IJField, N: gtx.int32) -> IJField:
    '''Periodic boundaries in J only; I is handled by the decomposition.'''
    f = concat_where(J == -1, f(J + N), f)
    f = concat_where(J == N, f(J - N), f)
    return f


@gtx.field_operator
def five_point(f: IJField) -> IJField:
    return f(I + 1) + f(I - 1) + f(J + 1) + f(J - 1) - 4.0 * f


DOM_G_INT, DOM_G_HALO = interior_domain(M, N), halo_domain(M, N)


def global_pipeline(a_int):
    padded = jnp.zeros((M + 2, N + 2)).at[1:-1, 1:-1].set(a_int)
    f = halo_exchange(fld(DOM_G_HALO, padded), M, N)
    return five_point(f)[DOM_G_INT].ndarray


rng = np.random.default_rng(0)
a = jnp.asarray(rng.standard_normal((M, N)))
w = jnp.asarray(rng.standard_normal((M, N)))

reference_out = global_pipeline(a)
reference_grad = jax.grad(lambda z: jnp.sum(global_pipeline(z) * w))(a)
print(f"reference output shape {reference_out.shape}, "
      f"grad norm {float(jnp.linalg.norm(reference_grad)):.6f}")

reference output shape (12, 5), grad norm 35.407217


## 2. Route A — `shard_map` and `ppermute`

`jax.lax.ppermute` sends each shard's data to another shard along a mesh axis. It is a
**linear** operation, and JAX ships a transpose rule for it: the transpose of a
permutation is the inverse permutation. So reverse-mode AD walks straight through a
collective without being told anything.

Inside `shard_map` each rank sees only its own block. The local halo columns arrive by
`ppermute`; the `J` direction is still periodic within the rank, so it stays a
`concat_where`. Then the ordinary GT4Py field operator runs on the local field.

In [3]:
mesh = Mesh(np.array(jax.devices()), ("x",))
perm_left = tuple((i, (i - 1) % D) for i in range(D))
perm_right = tuple((i, (i + 1) % D) for i in range(D))

DOM_L_HALO = gtx.domain({I: (-1, MLOC + 1), J: (-1, N + 1)})
DOM_L_INT = gtx.domain({I: (0, MLOC), J: (0, N)})


def local_pipeline(a_loc):
    '''Runs on one rank: a_loc is this rank interior block, shape (MLOC, N).'''
    left = jax.lax.ppermute(a_loc[-1:, :], "x", perm_right)   # my last row -> right nbr
    right = jax.lax.ppermute(a_loc[:1, :], "x", perm_left)    # my first row -> left nbr
    with_i = jnp.concatenate([left, a_loc, right], axis=0)
    with_ij = jnp.pad(with_i, ((0, 0), (1, 1)))
    f = periodic_j(fld(DOM_L_HALO, with_ij), N)
    return five_point(f)[DOM_L_INT].ndarray


sharded_pipeline = jax.jit(
    shard_map(local_pipeline, mesh=mesh, in_specs=P("x"), out_specs=P("x"))
)

out_a = sharded_pipeline(a)
grad_a = jax.grad(lambda z: jnp.sum(sharded_pipeline(z) * w))(a)

print(f"forward  max |sharded - reference| = {float(jnp.max(jnp.abs(out_a - reference_out))):.2e}")
print(f"gradient max |sharded - reference| = {float(jnp.max(jnp.abs(grad_a - reference_grad))):.2e}")

forward  max |sharded - reference| = 0.00e+00
gradient max |sharded - reference| = 8.88e-16


A distributed GT4Py field operator, differentiated across four devices, and **not one
line of adjoint code was written**. The reverse exchange exists — it runs as a
`ppermute` in the opposite direction with an accumulation — but JAX derived it.

Worth seeing on its own, because it is the entire basis of this route:

In [4]:
def ring_shift(v):
    return jax.lax.ppermute(v, "x", perm_right)


ring = jax.jit(shard_map(ring_shift, mesh=mesh, in_specs=P("x"), out_specs=P("x")))
v = jnp.arange(1.0, D + 1)
seed = jnp.array([1.0, 0.0, 0.0, 0.0])

transposed = jax.grad(lambda z: jnp.sum(ring(z) * seed))(v)

print("forward  ppermute(v) :", np.asarray(ring(v)), "  <- each rank sends right")
print("transpose applied to :", np.asarray(seed))
print("gives                :", np.asarray(transposed), "  <- the inverse permutation")

forward  ppermute(v) : [4. 1. 2. 3.]   <- each rank sends right
transpose applied to : [1. 0. 0. 0.]
gives                : [0. 0. 0. 1.]   <- the inverse permutation


## 3. Route B — MPI and `custom_vjp`

MPI is invisible to JAX. A `comm.Sendrecv` is a Python side effect on concrete
buffers, so tracing cannot follow it and there is no transpose rule to inherit. The
exchange has to be wrapped in `jax.custom_vjp`, and **you** write the backward pass.

What should it compute? Exactly what nb01 derived:

- each halo cotangent travels back to the rank that owns the cell it came from,
- it is **added** to that owner's cotangent, not assigned,
- the halo cotangent is then dropped, which is the zeroing.

To check that rule without MPI, here is the same decomposition emulated in one
process: the distributed state is an array of shape `(D, MLOC, N)`, and a rank shift
is a `roll` along the first axis. Writing it twice — once transparently, so JAX
differentiates it, and once opaquely with a hand-written backward — lets the two be
compared directly.

In [5]:
def exchange_transparent(state):
    '''(D, MLOC, N) -> (D, MLOC+2, N). JAX can see through this one.'''
    left = jnp.roll(state, 1, axis=0)[:, -1:, :]
    right = jnp.roll(state, -1, axis=0)[:, :1, :]
    return jnp.concatenate([left, state, right], axis=1)


@jax.custom_vjp
def exchange_opaque(state):
    return exchange_transparent(state)


def _fwd(state):
    return exchange_opaque(state), None


def _bwd(_, g):
    state_bar = g[:, 1:-1, :]                        # each rank's own interior part
    left_bar = jnp.roll(g[:, :1, :], -1, axis=0)     # left-halo ct -> left nbr's last row
    right_bar = jnp.roll(g[:, -1:, :], 1, axis=0)    # right-halo ct -> right nbr's first row
    state_bar = state_bar.at[:, -1:, :].add(left_bar)
    state_bar = state_bar.at[:, :1, :].add(right_bar)
    return (state_bar,)                              # halo cotangent dropped == zeroed


exchange_opaque.defvjp(_fwd, _bwd)


def distributed_pipeline(a_int, exchange):
    with_i = exchange(a_int.reshape(D, MLOC, N))
    blocks = []
    for r in range(D):
        padded = jnp.pad(with_i[r], ((0, 0), (1, 1)))
        f = periodic_j(fld(DOM_L_HALO, padded), N)
        blocks.append(five_point(f)[DOM_L_INT].ndarray)
    return jnp.concatenate(blocks, axis=0)


for name, ex in [("transparent", exchange_transparent), ("custom_vjp ", exchange_opaque)]:
    out = distributed_pipeline(a, ex)
    grd = jax.grad(lambda z: jnp.sum(distributed_pipeline(z, ex) * w))(a)
    print(f"{name}: forward {float(jnp.max(jnp.abs(out - reference_out))):.2e}"
          f"   gradient {float(jnp.max(jnp.abs(grd - reference_grad))):.2e}")

transparent: forward 0.00e+00   gradient 8.88e-16
custom_vjp : forward 0.00e+00   gradient 8.88e-16


The hand-written backward reproduces both JAX's own answer and the single-process
reference. And the dot-product test applies to the exchange in isolation:

In [6]:
xs = jnp.asarray(rng.standard_normal((D, MLOC, N)))
ys = jnp.asarray(rng.standard_normal((D, MLOC + 2, N)))

lhs = float(jnp.sum(exchange_opaque(xs) * ys))
(xs_bar,) = _bwd(None, ys)
rhs = float(jnp.sum(xs * xs_bar))
print(f"<Lx, y>    = {lhs:.16e}")
print(f"<x, L^T y> = {rhs:.16e}")
print(f"relative   = {abs(lhs - rhs) / abs(lhs):.2e}")

<Lx, y>    = 1.8758110024620982e+00
<x, L^T y> = 1.8758110024620975e+00
relative   = 3.55e-16


### The same thing with real MPI

`mpi_halo_exchange.py` in this directory is that rule written against `mpi4py`. The
index algebra is identical — only `roll` becomes `Sendrecv`:

| emulated | MPI |
|---|---|
| `roll(g[:, :1, :], -1)` then `+=` into last row | `Sendrecv(g[0:1], dest=LEFT, source=RIGHT)` then `+=` |
| `roll(g[:, -1:, :], +1)` then `+=` into first row | `Sendrecv(g[-1:], dest=RIGHT, source=LEFT)` then `+=` |
| slicing `g[:, 1:-1, :]` | not sending the halo back at all |

One wrinkle: MPI needs concrete buffers, so the calls go through
`jax.pure_callback`, which means the exchange cannot live under `jax.jit`.
`mpi4jax` solves this properly by registering MPI operations as XLA custom calls.

The script runs a distributed dot-product test with `MPI.SUM` allreduces, on four real
MPI ranks. (One machine-specific detail: this laptop needs `HWLOC_COMPONENTS=-gl`, because
hydra's hwloc topology probe otherwise blocks on a stale X display socket and `MPI_Init`
never returns. That cost a day to find; it is documented in the project status file.)

In [7]:
import sys

# HWLOC_COMPONENTS=-gl: on this machine hydra's hwloc probe blocks on a stale X display
# socket and MPI_Init never returns without it (see STATUS.md, "MPI: fixed").
env = dict(os.environ, HWLOC_COMPONENTS="-gl", JAX_PLATFORMS="cpu")
try:
    r = subprocess.run(["mpirun", "-n", "4", sys.executable, "mpi_halo_exchange.py"],
                       capture_output=True, text=True, timeout=180, env=env)
    print("\n".join(l for l in r.stdout.splitlines() if "NVIDIA" not in l) or "(no stdout)")
    if r.returncode != 0:
        print("stderr:", r.stderr[-800:])
except subprocess.TimeoutExpired:
    subprocess.run(["killall", "-q", "-9", "mpiexec.hydra", "hydra_pmi_proxy"])
    print("mpirun timed out -- MPI is not usable on this machine; the backward rule above\n"
          "was verified on the emulated decomposition, and this script is unverified here.")
except FileNotFoundError:
    print("no mpirun on PATH -- skipping")

ranks              : 4
<Lx, y>            : -1.1514570920346753e+01
<x, L^T y>         : -1.1514570920346753e+01
relative difference: 0.00e+00
PASS


Relative difference at roundoff, with the sum genuinely reduced across four processes.
The backward rule that nb01 derived and the emulated decomposition verified above is the
same one running here — only the transport changed.

## 4. Choosing between them

| | Route A: `shard_map` + `ppermute` | Route B: MPI + `custom_vjp` |
|---|---|---|
| adjoint code to write | none | the whole backward pass |
| correctness risk | JAX's transpose rule | yours |
| runs under `jit` | yes | no, without `mpi4jax` |
| multi-node | JAX's own distributed runtime | anything MPI reaches |
| fits existing HPC stacks | poorly — JAX owns the decomposition | directly: GHEX, ICON, MPI codes |
| halo shape | whatever the mesh axis gives you | arbitrary, incl. unstructured |

Route A is remarkable and nearly free, and it is the right answer if JAX is allowed to
own the domain decomposition. It is the wrong answer for a model whose decomposition,
communication schedule, and halo structure already exist in a Fortran or C++ stack —
which describes essentially every operational weather model, ICON included. There,
Route B is the only option, and nb01 is how you know your backward pass is right.

## 5. What is still not covered

- **Correctness of the schedule, not just the arithmetic.** These exchanges are
  blocking and ordered. Real codes overlap communication with computation and post
  many neighbour exchanges at once; the adjoint of an overlapped schedule has to
  reverse the ordering as well as the direction, and nothing here tests that.
- **The tape.** Every stage above keeps a full field. A distributed implementation
  wants to tape only halo buffers, and for a real model that difference is the
  difference between fitting in memory and not. `nb01` §13 makes the same point.
- **Unstructured halos.** A ring of contiguous rows is the easy case. ICON's halos
  are index lists, and the transpose of a gather through an index list is a
  scatter-add with duplicate targets — correct under JAX, but the performance
  question is completely open.
- **The compiled backends.** All of this is embedded mode. Switch to `gtfn` or
  `dace` and the gradient disappears, because JAX no longer sees the computation at
  all. Getting AD through a compiled GT4Py program is a different project: it needs
  a `custom_vjp` whose backward is a *generated* adjoint program, and the halo
  adjoint derived in nb01 is a small worked example of what that generator would
  have to emit.